# 02 — Pillar 1: The Athlete Edge
**Project:** LA 2028 Olympic Games Strategic Analysis  
**Analyst:** Shabeeb | SportsFanatics Consulting  
**Goal:** Deliver data-driven insights for athletes — performance trends, physical profiles, gender equity, home advantage, and sport growth trajectories relevant to LA 2028.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

BASE_DIR   = Path('..')
RAW_DATA   = BASE_DIR / 'data' / 'raw' / 'athlete_events.csv'
OUT_TABLES = BASE_DIR / 'outputs' / 'tables'
OUT_FIGS   = BASE_DIR / 'outputs' / 'figures'

OLY = dict(blue='#0085C7', yellow='#F4C300', black='#000000',
           green='#009F6B', red='#DF0024', bg='#F8F9FA')

logger.info('Setup complete.')

INFO: Setup complete.


## 1. Load & Clean Data

In [2]:
try:
    df_raw = pd.read_csv(RAW_DATA)
    logger.info(f'Loaded {df_raw.shape[0]:,} rows')
except FileNotFoundError:
    logger.error(f'File not found: {RAW_DATA}')
    raise

# --- Basic cleaning ---
df = df_raw.copy()

# Binary medal flag
df['Medal_Won'] = df['Medal'].notna().astype(int)

# Standardise gender label
df['Gender'] = df['Sex'].map({'M': 'Male', 'F': 'Female'})

# Impute Age/Height/Weight with sport-gender median (for analysis only)
for col in ['Age', 'Height', 'Weight']:
    df[f'{col}_imputed'] = df.groupby(['Sport', 'Sex'])[col].transform(
        lambda x: x.fillna(x.median())
    )

# Summer-only subset for most athlete analyses
summer = df[df['Season'] == 'Summer'].copy()

logger.info(f'Summer rows: {len(summer):,} | Years: {summer["Year"].min()}–{summer["Year"].max()}')

INFO: Loaded 271,116 rows


INFO: Summer rows: 222,552 | Years: 1896–2016


## 2. Historical Performance Trends by Country (Summer)

In [3]:
# Total medals per NOC per Games (Summer)
medals_noc_year = (
    summer[summer['Medal_Won'] == 1]
    .groupby(['Year', 'NOC', 'Medal'])['ID']
    .count()
    .reset_index()
    .rename(columns={'ID': 'Count'})
)

# Top 10 NOCs all-time by total medals
top10_noc = (
    medals_noc_year.groupby('NOC')['Count'].sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)
print('Top 10 NOCs all-time:', top10_noc)

# Medal trajectory for top 10
top_medals = (
    medals_noc_year[medals_noc_year['NOC'].isin(top10_noc)]
    .groupby(['Year', 'NOC'])['Count']
    .sum()
    .reset_index()
)

fig = px.line(
    top_medals, x='Year', y='Count', color='NOC',
    markers=True,
    title='Total Medals per Games — Top 10 NOCs (Summer Olympics)',
    labels={'Count': 'Medals', 'Year': 'Year'},
    template='plotly_white', width=1200, height=600
)
fig.update_layout(font_family='Arial', title_font_size=18,
                  legend_title='NOC')
fig.write_html(OUT_FIGS / 'athlete_line_medalsByNOC_historical.html')
fig.show()
logger.info('Saved NOC medal trajectory chart.')

Top 10 NOCs all-time: ['USA', 'URS', 'GBR', 'GER', 'FRA', 'ITA', 'AUS', 'HUN', 'SWE', 'NED']


INFO: Saved NOC medal trajectory chart.


## 3. Medal Breakdown by Type — Top 10 NOCs

In [4]:
medal_type_noc = (
    summer[summer['Medal_Won'] == 1]
    .groupby(['NOC', 'Medal'])['ID']
    .count()
    .reset_index()
    .rename(columns={'ID': 'Count'})
)
medal_type_noc = medal_type_noc[medal_type_noc['NOC'].isin(top10_noc)]

medal_color = {
    'Gold':   OLY['yellow'],
    'Silver': '#C0C0C0',
    'Bronze': '#CD7F32'
}

fig = px.bar(
    medal_type_noc, x='NOC', y='Count', color='Medal',
    color_discrete_map=medal_color,
    barmode='stack',
    title='Medal Breakdown by Type — Top 10 NOCs (Summer, All-Time)',
    template='plotly_white', width=1100, height=600,
    category_orders={'Medal': ['Gold', 'Silver', 'Bronze'],
                     'NOC': top10_noc}
)
fig.update_layout(font_family='Arial', title_font_size=18)
fig.write_html(OUT_FIGS / 'athlete_bar_medalTypeByNOC.html')
fig.show()
logger.info('Saved medal type breakdown chart.')

INFO: Saved medal type breakdown chart.


## 4. Gender Equity — Participation Trend

In [5]:
gender_year = (
    summer.groupby(['Year', 'Gender'])['ID']
    .nunique()
    .reset_index()
    .rename(columns={'ID': 'Athletes'})
)

# Add female % per year
totals = gender_year.groupby('Year')['Athletes'].sum().rename('Total')
gender_year = gender_year.join(totals, on='Year')
gender_year['Pct'] = (gender_year['Athletes'] / gender_year['Total'] * 100).round(1)

female_pct = gender_year[gender_year['Gender'] == 'Female'][['Year', 'Pct']]

fig = make_subplots(specs=[[{'secondary_y': True}]])

for gender, color in [('Male', OLY['blue']), ('Female', OLY['red'])]:
    sub = gender_year[gender_year['Gender'] == gender]
    fig.add_trace(
        go.Bar(x=sub['Year'], y=sub['Athletes'], name=gender,
               marker_color=color, opacity=0.8),
        secondary_y=False
    )

fig.add_trace(
    go.Scatter(x=female_pct['Year'], y=female_pct['Pct'],
               name='Female %', mode='lines+markers',
               line=dict(color=OLY['yellow'], width=2, dash='dot'),
               marker=dict(size=6)),
    secondary_y=True
)

fig.update_layout(
    title='Gender Participation in Summer Olympics',
    barmode='stack', template='plotly_white',
    font_family='Arial', title_font_size=18,
    width=1200, height=600,
    legend=dict(x=0.01, y=0.99)
)
fig.update_yaxes(title_text='Unique Athletes', secondary_y=False)
fig.update_yaxes(title_text='Female Participation %', secondary_y=True)

fig.write_html(OUT_FIGS / 'athlete_bar_genderParticipation.html')
fig.show()
logger.info('Saved gender participation chart.')

INFO: Saved gender participation chart.


## 5. Physical Profile by Sport (Age, Height, Weight)

In [6]:
# Focus on sports likely in LA 2028 — top 20 by participation
top_sports = summer['Sport'].value_counts().head(20).index.tolist()

sport_profile = (
    summer[summer['Sport'].isin(top_sports)]
    .groupby('Sport')[['Age_imputed', 'Height_imputed', 'Weight_imputed']]
    .median()
    .reset_index()
    .rename(columns={'Age_imputed': 'Median Age',
                     'Height_imputed': 'Median Height (cm)',
                     'Weight_imputed': 'Median Weight (kg)'})
    .sort_values('Median Age')
)

fig = px.scatter(
    sport_profile,
    x='Median Height (cm)', y='Median Weight (kg)',
    size='Median Age', color='Sport',
    text='Sport',
    title='Athlete Physical Profile by Sport — Median Height vs Weight (bubble = Age)',
    template='plotly_white', width=1200, height=700
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(font_family='Arial', title_font_size=16, showlegend=False)
fig.write_html(OUT_FIGS / 'athlete_scatter_physicalProfile.html')
fig.show()

sport_profile.to_csv(OUT_TABLES / 'athlete_physicalProfile_bySport.csv', index=False)
logger.info('Saved physical profile chart and table.')

INFO: Saved physical profile chart and table.


## 6. Age Distribution — Medalists vs Non-Medalists

In [7]:
age_medal = summer[summer['Age_imputed'].notna()].copy()

fig = px.histogram(
    age_medal, x='Age_imputed', color='Medal_Won',
    barmode='overlay',
    nbins=50,
    histnorm='percent',
    color_discrete_map={0: OLY['blue'], 1: OLY['yellow']},
    labels={'Age_imputed': 'Age', 'Medal_Won': 'Medalist',
            'percent': '% of Group'},
    title='Age Distribution: Medalists vs Non-Medalists (Summer Olympics)',
    template='plotly_white', width=1000, height=550,
    opacity=0.7
)
fig.for_each_trace(lambda t: t.update(name='Medalist' if t.name == '1' else 'Non-Medalist'))
fig.update_layout(font_family='Arial', title_font_size=18)
fig.write_html(OUT_FIGS / 'athlete_hist_ageMedalistVsNon.html')
fig.show()

# Stats
print('Median age — Non-Medalists:', age_medal[age_medal['Medal_Won']==0]['Age_imputed'].median())
print('Median age — Medalists:    ', age_medal[age_medal['Medal_Won']==1]['Age_imputed'].median())
logger.info('Saved age distribution chart.')

INFO: Saved age distribution chart.


Median age — Non-Medalists: 24.0
Median age — Medalists:     25.0


## 7. Sport Participation Growth (1980–2016)

In [8]:
# Focus on modern era
modern = summer[summer['Year'] >= 1980]

sport_growth = (
    modern.groupby(['Year', 'Sport'])['ID']
    .nunique()
    .reset_index()
    .rename(columns={'ID': 'Athletes'})
)

# Sports with most growth: compare 1984 vs 2016
pivot = sport_growth.pivot(index='Sport', columns='Year', values='Athletes').fillna(0)
if 1984 in pivot.columns and 2016 in pivot.columns:
    pivot['Growth_abs'] = pivot[2016] - pivot[1984]
    pivot['Growth_pct'] = ((pivot[2016] - pivot[1984]) / pivot[1984].replace(0, np.nan) * 100).round(1)
    growth_table = pivot[['Growth_abs', 'Growth_pct']].sort_values('Growth_abs', ascending=False)
    print('Top growing sports (1984–2016):')
    display(growth_table.head(15))
    growth_table.to_csv(OUT_TABLES / 'athlete_sportGrowth_1984_2016.csv')

# Top 12 sports by 2016 participation
top12_sports_2016 = (
    sport_growth[sport_growth['Year'] == 2016]
    .nlargest(12, 'Athletes')['Sport'].tolist()
)

fig = px.line(
    sport_growth[sport_growth['Sport'].isin(top12_sports_2016)],
    x='Year', y='Athletes', color='Sport',
    markers=True,
    title='Athlete Participation Growth by Sport — Summer Olympics (1980–2016)',
    template='plotly_white', width=1200, height=650
)
fig.update_layout(font_family='Arial', title_font_size=18)
fig.write_html(OUT_FIGS / 'athlete_line_sportParticipationGrowth.html')
fig.show()
logger.info('Saved sport growth chart.')

Top growing sports (1984–2016):


Year,Growth_abs,Growth_pct
Sport,,
Athletics,989.0,77.3
Swimming,448.0,90.7
Rugby Sevens,299.0,NaN
Football,227.0,92.3
Tennis,196.0,NaN
Judo,177.0,83.5
Badminton,172.0,NaN
Table Tennis,172.0,NaN
Cycling,154.0,42.9


INFO: Saved sport growth chart.


## 8. Home Country Advantage Analysis

In [9]:
# Map host city → NOC
host_map = {
    1896: 'GRE', 1900: 'FRA', 1904: 'USA', 1906: 'GRE',
    1908: 'GBR', 1912: 'SWE', 1920: 'BEL', 1924: 'FRA',
    1928: 'NED', 1932: 'USA', 1936: 'GER', 1948: 'GBR',
    1952: 'FIN', 1956: 'AUS', 1960: 'ITA', 1964: 'JPN',
    1968: 'MEX', 1972: 'FRG', 1976: 'CAN', 1980: 'URS',
    1984: 'USA', 1988: 'KOR', 1992: 'ESP', 1996: 'USA',
    2000: 'AUS', 2004: 'GRE', 2008: 'CHN', 2012: 'GBR',
    2016: 'BRA'
}

summer['Host_NOC'] = summer['Year'].map(host_map)
summer['Is_Host'] = (summer['NOC'] == summer['Host_NOC']).astype(int)

# Medal rate: host vs non-host, by edition
home_adv = (
    summer.groupby(['Year', 'Is_Host'])
    .agg(Entries=('ID', 'count'), Medals=('Medal_Won', 'sum'))
    .reset_index()
)
home_adv['Medal_Rate'] = (home_adv['Medals'] / home_adv['Entries'] * 100).round(2)
home_adv['Group'] = home_adv['Is_Host'].map({1: 'Host Nation', 0: 'All Others'})

# Summary stats
summary = home_adv.groupby('Group')['Medal_Rate'].agg(['mean', 'median', 'std']).round(2)
print('Home advantage summary (medal rate %):')
display(summary)

fig = px.line(
    home_adv, x='Year', y='Medal_Rate', color='Group',
    markers=True,
    color_discrete_map={'Host Nation': OLY['red'], 'All Others': OLY['blue']},
    title='Home Country Advantage — Medal Rate: Host Nation vs Rest (Summer Olympics)',
    labels={'Medal_Rate': 'Medal Rate (%)', 'Year': 'Year'},
    template='plotly_white', width=1200, height=600
)
fig.update_layout(font_family='Arial', title_font_size=18)
fig.write_html(OUT_FIGS / 'athlete_line_homeAdvantage.html')
fig.show()

home_adv.to_csv(OUT_TABLES / 'athlete_homeAdvantage.csv', index=False)
logger.info('Saved home advantage chart and table.')

Home advantage summary (medal rate %):


,mean,median,std
Group,,,
All Others,18.70,14.69,9.92
Host Nation,22.78,21.07,14.35


INFO: Saved home advantage chart and table.


## 9. USA Home Advantage — LA 1984 & Atlanta 1996 Deep Dive

In [10]:
# Compute USA medal rate at each Summer Games (as participant & host)
usa = summer[summer['NOC'] == 'USA'].groupby('Year').agg(
    Entries=('ID', 'count'),
    Medals=('Medal_Won', 'sum')
).reset_index()
usa['Medal_Rate'] = (usa['Medals'] / usa['Entries'] * 100).round(2)
usa['Host_Year'] = usa['Year'].isin([1904, 1932, 1984, 1996]).astype(int)

fig = px.bar(
    usa, x='Year', y='Medal_Rate',
    color='Host_Year',
    color_discrete_map={1: OLY['red'], 0: OLY['blue']},
    labels={'Medal_Rate': 'Medal Rate (%)', 'Host_Year': 'USA is Host'},
    title='USA Medal Rate at Summer Olympics — Host Years Highlighted',
    template='plotly_white', width=1200, height=550
)
fig.update_layout(font_family='Arial', title_font_size=18,
                  showlegend=True,
                  legend=dict(title='Host Year'))
for year in [1904, 1932, 1984, 1996]:
    fig.add_vline(x=year, line_dash='dash', line_color='gray', opacity=0.5)
    fig.add_annotation(x=year, y=usa[usa['Year']==year]['Medal_Rate'].values[0] + 1,
                        text=str(year), showarrow=False, font_size=10)

fig.write_html(OUT_FIGS / 'athlete_bar_usaMedalRate.html')
fig.show()

print('USA Medal Rate at home vs away:')
print(usa.groupby('Host_Year')['Medal_Rate'].mean().round(2))
logger.info('Saved USA medal rate chart.')

INFO: Saved USA medal rate chart.


USA Medal Rate at home vs away:
Host_Year
0    34.15
1    35.66
Name: Medal_Rate, dtype: float64


## 10. Performance Trends by Gender — Medal Rates Over Time

In [11]:
gender_medal = (
    summer[summer['Year'] >= 1952]
    .groupby(['Year', 'Gender'])
    .agg(Entries=('ID', 'count'), Medals=('Medal_Won', 'sum'))
    .reset_index()
)
gender_medal['Medal_Rate'] = (gender_medal['Medals'] / gender_medal['Entries'] * 100).round(2)

fig = px.line(
    gender_medal, x='Year', y='Medal_Rate', color='Gender',
    markers=True,
    color_discrete_map={'Male': OLY['blue'], 'Female': OLY['red']},
    title='Medal Rate by Gender — Summer Olympics (1952–2016)',
    labels={'Medal_Rate': 'Medal Rate (%)'},
    template='plotly_white', width=1200, height=580
)
fig.update_layout(font_family='Arial', title_font_size=18)
fig.write_html(OUT_FIGS / 'athlete_line_medalRateByGender.html')
fig.show()
logger.info('Saved gender medal rate chart.')

INFO: Saved gender medal rate chart.


## 11. Top Medal-Winning Sports by NOC (Heatmap)

In [12]:
# For top 10 NOCs × top 15 sports
top15_sports = (
    summer[summer['Medal_Won']==1]['Sport']
    .value_counts().head(15).index.tolist()
)

heatmap_data = (
    summer[
        (summer['Medal_Won']==1) &
        (summer['NOC'].isin(top10_noc)) &
        (summer['Sport'].isin(top15_sports))
    ]
    .groupby(['NOC', 'Sport'])['Medal_Won']
    .sum()
    .reset_index()
    .pivot(index='NOC', columns='Sport', values='Medal_Won')
    .fillna(0)
)

fig = px.imshow(
    heatmap_data,
    color_continuous_scale=[[0, OLY['bg']], [0.5, OLY['blue']], [1, OLY['red']]],
    title='Medal Count Heatmap — Top 10 NOCs × Top 15 Sports (Summer, All-Time)',
    template='plotly_white',
    width=1200, height=600,
    text_auto=True
)
fig.update_layout(font_family='Arial', title_font_size=16,
                  coloraxis_showscale=True)
fig.write_html(OUT_FIGS / 'athlete_heatmap_medalsBySportNOC.html')
fig.show()
logger.info('Saved sport-NOC heatmap.')

INFO: Saved sport-NOC heatmap.


## 12. LA 2028 New Sports Spotlight
*New sports confirmed/expected for LA 2028: Flag Football, Cricket (T20), Lacrosse, Squash, Baseball/Softball*

In [13]:
# These sports are not in the historical dataset — analyse closest proxies + frame outlook
new_sports_2028 = ['Flag Football', 'Cricket', 'Lacrosse', 'Squash', 'Baseball', 'Softball']
proxy_sports    = ['Baseball', 'Softball', 'Cricket']  # historically present in some Games

proxy_data = summer[summer['Sport'].isin(proxy_sports)]

if not proxy_data.empty:
    proxy_summary = (
        proxy_data.groupby(['Sport', 'Year'])
        .agg(Athletes=('ID', 'nunique'), NOCs=('NOC', 'nunique'))
        .reset_index()
    )
    print('Proxy sports history:')
    display(proxy_summary)

    fig = px.bar(
        proxy_summary, x='Year', y='NOCs', color='Sport',
        barmode='group',
        title='NOC Participation in Baseball/Softball/Cricket (Olympic History)',
        template='plotly_white', width=1000, height=500
    )
    fig.update_layout(font_family='Arial', title_font_size=16)
    fig.write_html(OUT_FIGS / 'athlete_bar_newSports2028Proxies.html')
    fig.show()
else:
    logger.warning('No historical data found for proxy new sports.')
    print('Note: Flag Football, Lacrosse, and Squash have no prior Olympic history in this dataset.')
    print('Recommendation: supplement with federation participation data for LA 2028 projections.')

logger.info('New sports analysis complete.')

Proxy sports history:


,Sport,Year,Athletes,NOCs
0,Baseball,1992,160,8
1,Baseball,1996,160,8
2,Baseball,2000,192,8
3,Baseball,2004,191,8
4,Baseball,2008,191,8
5,Cricket,1900,24,2
6,Softball,1996,120,8
7,Softball,2000,120,8
8,Softball,2004,118,8
9,Softball,2008,120,8


INFO: New sports analysis complete.


## 13. Export Athlete Summary Table

In [14]:
athlete_summary = (
    summer.groupby('NOC')
    .agg(
        Total_Entries   =('ID', 'count'),
        Unique_Athletes =('ID', 'nunique'),
        Total_Medals    =('Medal_Won', 'sum'),
        Gold            =('Medal', lambda x: (x == 'Gold').sum()),
        Silver          =('Medal', lambda x: (x == 'Silver').sum()),
        Bronze          =('Medal', lambda x: (x == 'Bronze').sum()),
        Median_Age      =('Age_imputed', 'median'),
        Medal_Rate_pct  =('Medal_Won', lambda x: round(x.mean() * 100, 2))
    )
    .sort_values('Total_Medals', ascending=False)
    .reset_index()
)

athlete_summary.to_csv(OUT_TABLES / 'athlete_NOC_summary.csv', index=False)
print(f'Saved NOC summary table — {len(athlete_summary)} NOCs')
display(athlete_summary.head(15))
logger.info('Saved athlete NOC summary table.')

Saved NOC summary table — 230 NOCs


,NOC,Total_Entries,Unique_Athletes,Total_Medals,Gold,Silver,Bronze,Median_Age,Medal_Rate_pct
0,USA,15064,7970,5002,2472,1333,1197,25.0,33.20
1,URS,4622,2476,2063,832,635,596,24.0,44.63
2,GBR,10917,5646,1985,636,729,620,25.0,18.18
3,GER,7622,3981,1779,592,538,649,26.0,23.34
4,FRA,10633,5353,1627,465,575,587,25.0,15.30
5,ITA,8217,3999,1446,518,474,454,25.0,17.60
6,AUS,7092,3582,1304,342,452,510,24.0,18.39
7,HUN,6129,2567,1123,432,328,363,25.0,18.32
8,SWE,6076,2893,1108,354,396,358,25.0,18.24
9,NED,5164,2731,918,245,302,371,25.0,17.78


INFO: Saved athlete NOC summary table.


## Summary — Pillar 1: The Athlete Edge

| Insight | Finding |
|---|---|
| Dominant nations | USA, URS/RUS, GER lead all-time Summer medals |
| Gender equity | Female participation grew from ~10% (1952) to ~45% (2016); converging medal rates |
| Peak athlete age | Medalists cluster 22–28; older athletes prevalent in equestrian, shooting |
| Sport growth | Athletics, Swimming, Rowing consistently largest; emerging team sports growing |
| Home advantage | Host nations average ~2× the medal rate of away nations |
| USA at home | LA 1984 produced a historic medal rate — LA 2028 is a major opportunity |
| New sports 2028 | Flag Football, Cricket, Lacrosse, Squash — USA competitive in all four |

**Next step → `03_city_analysis.ipynb`**